# Part 3: Give it ears (and a mouth)

Whisper runs on your laptop as its own little model. The LLM never hears your
voice, only the text whisper wrote down. That's a design choice, and at the end
you can try the other one.

No mic? Every cell that records has a line to load a `.wav` file instead.

In [ ]:
# Run me first (Shift+Enter). Re-run me if you restart the kernel.
%load_ext autoreload
%autoreload 2

import config
from agent import nb, ears, mouth
from agent.llm import LocalLLM, system, user, assistant

llm = LocalLLM()
print("brain:", llm.model_name() if llm.health() else f"NOT RUNNING at {config.SERVER_URL} (run the Step 0 cells in 1_setup.ipynb)")

In [ ]:
import time

WHISPER_MODEL = "base.en"   # "tiny.en" (fastest) | "base.en" | "small.en" (most accurate)
VOCAB = None                # e.g. "llama.cpp, Gemma, Arm, Pier 48" nudges the spelling

t0 = time.time()
ears.load_whisper(WHISPER_MODEL)
print(f"whisper {WHISPER_MODEL} loaded in {time.time() - t0:.1f}s")

## Record, listen, transcribe

**x realtime** = seconds of speech per second of work. Above 1x means it keeps
up with you talking.

In [ ]:
audio = nb.record(4)                       # or: audio = ears.load_wav("my_clip.wav")
nb.play(audio)

text, secs = ears.transcribe(audio, model=WHISPER_MODEL, vocab=VOCAB)
speech_s = len(audio) / config.SAMPLE_RATE
print(f"heard: {text!r}")
print(f"{speech_s:.1f}s of audio in {secs:.2f}s = {speech_s / max(secs, 1e-6):.1f}x realtime")

## Talk to it

Now the transcript goes to the brain, and the answer is spoken **sentence by
sentence while it's still being written**. That trick (`StreamingMouth`) is most
of why voice assistants feel fast.

Run the cell, talk, repeat. The conversation carries over between runs.

In [ ]:
VOICE_PROMPT = ("You are a voice assistant. Your answer will be read aloud, so: one to three "
                "short sentences, no lists, no markdown, no emoji.")
talk_history = [system(VOICE_PROMPT)]

def talk(seconds=4):
    audio = nb.record(seconds)                      # or ears.load_wav("clip.wav")
    said, stt_s = ears.transcribe(audio, model=WHISPER_MODEL, vocab=VOCAB)
    print(f"you: {said}")
    if not said:
        return
    talk_history.append(user(said))
    voice = mouth.StreamingMouth()
    def on_token(piece):
        nb.stream(piece)
        voice.feed(piece)
    print("bot: ", end="")
    r = llm.chat(talk_history, on_token=on_token, max_tokens=150)
    talk_history.append(assistant(r.text))
    print(f"\n\nstopped talking -> first word: {stt_s + (r.first_token_s or 0):.2f}s "
          f"(whisper {stt_s:.2f}s + brain {r.first_token_s or 0:.2f}s)")
    voice.finish()

talk()

In [ ]:
talk()   # again! it remembers the conversation

## Whisper size shoot-out

Same clip, three models. The first run of each downloads it (75 to 480 MB),
so maybe skip `small.en` on conference wifi.

In [ ]:
for m in ("tiny.en", "base.en", "small.en"):
    t0 = time.time()
    ears.load_whisper(m)
    load_s = time.time() - t0
    text, secs = ears.transcribe(audio, model=m)
    print(f"{m:>9}: {secs:5.2f}s  (+{load_s:.1f}s load)  {text!r}")
ears.load_whisper(WHISPER_MODEL)   # back to your pick

## Your turn: a wake word

Only answer if the transcript starts with **"computer"** (hello, Star Trek). Fill in the TODO, then run.
What does whisper do with it when you mumble? What about "commuter"?

In [ ]:
WAKE_WORD = "computer"

def heard_wake_word(transcript: str) -> bool:
    # TODO: return True if the transcript starts with the wake word.
    #       Hint: .lower(), .strip(" ,.!?"), .startswith()
    return True

audio = nb.record(4)
said, _ = ears.transcribe(audio, model=WHISPER_MODEL)
print(repr(said), "->", "ANSWER" if heard_wake_word(said) else "ignore")

*Stuck? The cell below is a solution. Click the `...` to reveal it.*

In [ ]:
# SOLUTION
def heard_wake_word(transcript: str) -> bool:
    words = transcript.lower().replace(",", " ").split()
    return bool(words) and words[0].strip(" .!?") in (WAKE_WORD, "computers", "commuter")

print(repr(said), "->", "ANSWER" if heard_wake_word(said) else "ignore")

## Your turn: "sorry, say that again?"

Whisper returns confident text even for mumbling. Each segment has an
`avg_logprob`: closer to 0 is more sure. Mumble, then speak clearly, and
compare. Pick a threshold below which your assistant should ask you to repeat.

In [ ]:
w = ears.load_whisper(WHISPER_MODEL)
audio = nb.record(4)
segments, _ = w.transcribe(audio, language="en", beam_size=1, vad_filter=True)
for s in segments:
    print(f"{s.avg_logprob:6.2f}  {s.text!r}")

THRESHOLD = -1.0   # TODO: tune me

## Bonus: skip whisper entirely

Gemma 4 (E2B/E4B) can take audio directly. One model, one call, no transcript
step. Only works if that's the model you loaded. Faster? Slower? More accurate?

In [ ]:
from agent.llm import audio_part

clip = nb.record(4)
t0 = time.time()
r = llm.chat([{"role": "user", "content": [
                  audio_part(ears.to_wav_bytes(clip)),
                  {"type": "text", "text": "Transcribe this audio exactly. Output only the words."}]}],
             stream=False, max_tokens=200, temperature=0)
print(f"llm heard : {r.text.strip()!r} in {time.time() - t0:.2f}s")
text, secs = ears.transcribe(clip, model=WHISPER_MODEL)
print(f"whisper   : {text!r} in {secs:.2f}s")

## Try this

1. Say "llama.cpp" and your surname. Put them in `VOCAB`, reload whisper, try again.
2. Record 5 seconds of silence. Whisper sometimes "hears" *"Thank you for watching."*
   in silence (it learned from subtitled videos). We set `vad_filter=True` to trim silence first.
3. In `talk()`, where does the time go: whisper or the brain? Which knob would you turn first?